In [1]:
# ============================================================
# Project: NDVI Mapping of Andhra Pradesh using Sentinel-2
# Platform: Google Earth Engine + Google Colab
# ============================================================

# ---------------------------
# Import Libraries
# ---------------------------
import os
import ee
import geemap

# ---------------------------
# Authenticate and Initialize Earth Engine
# ---------------------------
ee.Authenticate()
ee.Initialize(project='spatial-12345')

# ---------------------------
# Create Output Folder
# ---------------------------
output_folder = "outputs"
os.makedirs(output_folder, exist_ok=True)

# ---------------------------
# Create Interactive Map
# ---------------------------
Map = geemap.Map()

# ---------------------------
# Define Region of Interest (ROI)
# Andhra Pradesh (Approximate Bounding Box)
# ---------------------------
roi = ee.FeatureCollection(
    ee.Geometry.Rectangle([
        76.75, 12.50,
        84.70, 19.50
    ])
)

Map.addLayer(roi, {}, "Andhra Pradesh ROI")
Map.centerObject(roi, 7)

# ---------------------------
# Load Sentinel-2 Surface Reflectance Images
# ---------------------------
sentinel2 = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(roi)
    .filterDate("2025-01-01", "2025-12-31")
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 10))
)

image = sentinel2.median().clip(roi)

# ---------------------------
# True Colour Visualization
# ---------------------------
true_colour = {
    "bands": ["B4", "B3", "B2"],
    "min": 0,
    "max": 2000,
    "gamma": 1
}

Map.addLayer(image, true_colour, "True Colour")

# ---------------------------
# Calculate NDVI
# NDVI = (NIR - RED) / (NIR + RED)
# ---------------------------
ndvi = image.normalizedDifference(["B8", "B4"]).rename("NDVI")

# ---------------------------
# NDVI Colour Palette
# ---------------------------
ndvi_palette = [
    "ffffff",
    "ce7e45",
    "f1b555",
    "fcd163",
    "99b718",
    "74a901",
    "66a000",
    "529400",
    "3e8601",
    "207401",
    "056201",
    "004c00",
    "023b01",
    "012e01",
    "011d01",
    "011301",
]

Map.addLayer(
    ndvi,
    {
        "min": 0,
        "max": 1,
        "palette": ndvi_palette
    },
    "NDVI"
)

# ---------------------------
# Display Map
# ---------------------------
Map

# ---------------------------
# Save Interactive Map as HTML
# ---------------------------
map_path = os.path.join(output_folder, "NDVI_Map.html")
Map.to_html(map_path)

print(f"Interactive map saved to: {map_path}")

# ---------------------------
# Export NDVI Raster to Google Drive
# ---------------------------
export_task = ee.batch.Export.image.toDrive(
    image=ndvi,
    description="NDVI_AndhraPradesh_Sentinel2",
    folder="earthengine_exports",
    fileNamePrefix="ndvi_andhra_pradesh",
    region=roi.geometry(),
    scale=100,
    maxPixels=1e10
)

export_task.start()

print("NDVI export started.")
print("Check the Earth Engine Tasks tab and your Google Drive for the exported GeoTIFF.")

Interactive map saved to: outputs/NDVI_Map.html
NDVI export started.
Check the Earth Engine Tasks tab and your Google Drive for the exported GeoTIFF.
